# Guía PC2 · Regresión y series de tiempo en Colab

**Cambia los datos del enunciado y sigue tu ruta.** El código principal usa el mismo estilo de RapiMenu, agentes y recaudación: preparar, entrenar un modelo por sección, comparar, interpretar y predecir. Los hiperparámetros ya están escritos junto a cada algoritmo.

| Tiempo orientativo | Trabajo |
|---|---|
| 10 min | Cargar y configurar |
| 20 min | Calidad, limpieza y EDA |
| 15 min | Preparación y división |
| 30 min | Modelos y comparación |
| 20 min | Evaluación e interpretación |
| 15 min | Predicción y entrega |

**Ruta:** 1–3 comunes → **4 regresión** o **5 series** → 6 evaluación → 7 predicción. Los bloques del otro problema se omiten. Los anexos al final están apagados; clasificación queda allí como apoyo. Los tiempos son una guía para organizar tus 110 minutos, no una medición de tu base.

Sube este notebook y tu archivo al panel Archivos de Colab. Ejecuta la instalación, completa la configuración y continúa en orden. Si cambias de base o problema, reinicia y ejecuta desde el inicio. No necesitas instalar Python en tu PC.

## 1. Entorno y configuración
Ejecuta la instalación una vez. Si Colab pide reiniciar la sesión, reinicia y vuelve a importar. No se instalan AutoML, TensorFlow ni SHAP durante el recorrido básico.

In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<2.3" "scikit-learn>=1.5,<1.7" "matplotlib>=3.8,<4" "seaborn>=0.13,<1" "xgboost>=2.1.4,<3" "lightgbm>=4.6,<5" openpyxl

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
sns.set_theme(style="whitegrid")

### Cambia solo los datos de tu enunciado
En tabular, excluye IDs, fechas sin transformar y datos conocidos después del resultado. Los demás predictores se detectan por tipo. Ejemplo RapiMenu: objetivo tiempo_entrega_min y exclusiones ["id_pedido", "fecha_hora"].

Las últimas cuatro líneas se usan únicamente para series. Un horizonte de 21 en frecuencia D son 21 días. Para clasificar, selecciona clasificacion y continúa con el anexo A.

In [ ]:
RUTA_ARCHIVO = "/content/mi_base.csv"
TIPO_PROBLEMA = "regresion"  # "regresion", "serie_tiempo" o "clasificacion"
TARGET = "CAMBIA_POR_TU_TARGET"
COLUMNAS_EXCLUIR = []  # IDs y variables que no debes usar para predecir

# SOLO SERIES DE TIEMPO
FECHA = "fecha"
FORMATO_FECHA = "%Y-%m-%d"  # "%d/%m/%Y" si tu archivo usa día/mes/año
FRECUENCIA = "D"  # "D": diaria, "W-MON": semanal, "MS": mensual
HORIZONTE = 21

## 2. Carga y calidad
La función leer_archivo(ruta) reconoce CSV/Excel; CSV usa detección de separador y UTF-8, Excel la primera hoja. Si tu CSV requiere latin1 o coma decimal, cambia esos argumentos en esta función. Las mismas reglas se usarán para la base nueva.

In [ ]:
def leer_archivo(ruta):
    """Lee tu CSV o Excel con las mismas opciones en todas las etapas."""
    archivo = Path(ruta)
    if archivo.suffix.lower() == ".xlsx":
        return pd.read_excel(archivo)
    if archivo.suffix.lower() == ".csv":
        return pd.read_csv(archivo, sep=None, engine="python", encoding="utf-8-sig", decimal=".")
    raise ValueError("Usa un archivo .csv o .xlsx.")

df_original = leer_archivo(RUTA_ARCHIVO)
if df_original.empty or not df_original.columns.is_unique:
    raise ValueError("La base está vacía o tiene columnas repetidas.")
if TIPO_PROBLEMA not in {"regresion", "serie_tiempo", "clasificacion"}:
    raise ValueError("Elige regresion, serie_tiempo o clasificacion.")
if TARGET not in df_original or set(COLUMNAS_EXCLUIR) - set(df_original.columns):
    raise ValueError("Revisa TARGET y COLUMNAS_EXCLUIR: deben existir en el archivo.")
if TIPO_PROBLEMA == "serie_tiempo" and FECHA not in df_original:
    raise ValueError("La columna FECHA no existe en tu archivo.")
print("Filas y columnas:", df_original.shape)
display(df_original.head())
display(pd.DataFrame({"tipo": df_original.dtypes, "nulos": df_original.isna().sum(),
                      "nulos_pct": df_original.isna().mean().mul(100)}))
print("Filas duplicadas:", df_original.duplicated().sum())

### Corrige solo lo que hayas identificado
preparar_datos(datos) conserva la entrada y concentra tus reglas, como limpiar/crear_variables en las referencias. Por defecto no elimina filas ni cambia valores. Puedes escribir reemplazos o conversiones aquí; se reutilizan al predecir otra base. Si agregas una regla que necesita el target, aplícala solo cuando esa columna exista.

Los nulos de predictores se imputarán después de dividir; **el target no se imputa**. Un atípico no se elimina solo por ser grande.

In [ ]:
def preparar_datos(datos):
    """Copia los datos y aplica únicamente las correcciones que escribas aquí."""
    tabla = datos.copy()
    # Ejemplo: tabla = tabla.replace({"antiguedad_meses": {999: np.nan}})
    # Ejemplo: tabla = tabla.assign(edad=pd.to_numeric(tabla["edad"], errors="raise"))
    # Ejemplo: tabla = tabla.assign(fin_de_semana=tabla["dia_semana"].isin(["Sábado", "Domingo"]).astype(int))
    return tabla

df_limpio = preparar_datos(df_original)
if df_limpio.empty or TARGET not in df_limpio or df_limpio[TARGET].isna().any():
    raise ValueError("La base quedó vacía o el target tiene nulos. Resuélvelos explícitamente.")
if TIPO_PROBLEMA != "clasificacion" and (not pd.api.types.is_numeric_dtype(df_limpio[TARGET])
        or not np.isfinite(df_limpio[TARGET].to_numpy(dtype=float)).all()):
    raise ValueError("El target debe ser numérico y finito. Revisa las conversiones.")

## 3. EDA: evidencia para tus respuestas
Revisa distribución del objetivo, posibles valores extraños, relaciones y diferencias entre grupos. Se limitan los gráficos para ahorrar tiempo. El EDA previo describe la base disponible; la preparación del modelo se aprende después con entrenamiento.

Anota dos hallazgos: **patrón + evidencia numérica + implicación**. No copies las cifras de los casos de referencia.

In [ ]:
display(df_limpio.describe(include="all").T)
if TIPO_PROBLEMA == "serie_tiempo":
    fechas_eda = pd.to_datetime(df_limpio[FECHA], format=FORMATO_FECHA, errors="raise")
    pd.Series(df_limpio[TARGET].to_numpy(), index=fechas_eda).sort_index().plot(figsize=(12, 4), title="Serie histórica")
    plt.show()
    display(df_limpio.assign(dia_semana=fechas_eda.dt.dayofweek).groupby("dia_semana")[TARGET].agg(["count", "mean", "median"]))
else:
    if TIPO_PROBLEMA == "regresion":
        sns.histplot(df_limpio[TARGET], bins=30)
    else:
        df_limpio[TARGET].value_counts().plot.bar()
    plt.title("Distribución del objetivo")
    plt.show()
    datos_eda = df_limpio.drop(columns=COLUMNAS_EXCLUIR)
    numericas_eda = datos_eda.select_dtypes(include="number").columns
    if len(numericas_eda) > 1:
        sns.heatmap(datos_eda[numericas_eda].corr(), cmap="coolwarm", center=0)
        plt.title("Correlaciones numéricas")
        plt.show()
    categoricas_eda = datos_eda.select_dtypes(include=["object", "category", "bool"]).columns
    for columna in [c for c in categoricas_eda if c != TARGET][:3]:
        if TIPO_PROBLEMA == "regresion":
            display(df_limpio.groupby(columna, dropna=False)[TARGET].agg(["count", "mean", "median"]).head(15))
        else:
            display(pd.crosstab(df_limpio[columna], df_limpio[TARGET], normalize="index").head(15))

### Herramientas comunes: ejecutar sin editar
metricas(reales, predicciones) devuelve MAE, RMSE, MAPE y R². MAPE excluye targets iguales a cero e informa cuántos; con todos ceros devuelve NaN. Menor MAE/RMSE/MAPE es mejor; R² puede ser negativo.

crear_preparacion(X) detecta cantidades y categorías. El Pipeline imputa con mediana/moda, escala cantidades y codifica categorías nuevas sin crear columnas a mano. Se ajusta solo con los datos con que se entrena cada modelo.

In [ ]:
def metricas(reales, predicciones):
    """Calcula errores; MAPE excluye ceros y R2 necesita al menos dos casos."""
    real, pred = np.asarray(reales, dtype=float), np.asarray(predicciones, dtype=float)
    no_cero = real != 0
    return {"MAE": mean_absolute_error(real, pred), "RMSE": np.sqrt(mean_squared_error(real, pred)),
            "MAPE_pct": np.mean(np.abs((real[no_cero] - pred[no_cero]) / real[no_cero])) * 100 if no_cero.any() else np.nan,
            "R2": r2_score(real, pred) if len(real) > 1 else np.nan, "ceros_MAPE": int((~no_cero).sum())}


def crear_preparacion(X):
    """Construye transformaciones sin ajustar; el modelo las aprende con train."""
    numericas = X.select_dtypes(include="number").columns.tolist()
    categoricas = [c for c in X.columns if c not in numericas]
    return ColumnTransformer([
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median", keep_empty_features=True)),
                          ("escalar", StandardScaler())]), numericas),
        ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent", keep_empty_features=True)),
                          ("codificar", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categoricas)])

modelos = {}
SEMILLA = 42

## 4. Ruta de regresión
### 4.1 Preparar y dividir
Sigue RapiMenu y agentes: separa X/y y entrena modelos sobre los mismos predictores. Se conserva 60% train, 20% validación y 20% prueba para elegir sin usar prueba. Los nombres detectados se muestran: verifica especialmente IDs y códigos numéricos que representen categorías.

Una fecha ya convertida se excluye; una fecha leída como texto debe estar en COLUMNAS_EXCLUIR o transformarse en preparar_datos. Los nulos se conservan hasta el Pipeline.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    X = df_limpio.drop(columns=[TARGET] + COLUMNAS_EXCLUIR).select_dtypes(exclude=["datetime", "datetimetz", "timedelta"])
    y = df_limpio[TARGET]
    if X.empty or np.isinf(X.select_dtypes(include="number").to_numpy(dtype=float, na_value=np.nan)).any():
        raise ValueError("Revisa los predictores: no pueden estar vacíos ni contener infinitos.")
    print("Predictores:", X.columns.tolist())
    X_desarrollo, X_prueba, y_desarrollo, y_prueba = train_test_split(X, y, test_size=0.20, random_state=SEMILLA)
    X_train, X_validacion, y_train, y_validacion = train_test_split(X_desarrollo, y_desarrollo, test_size=0.25, random_state=SEMILLA)
    preparacion = crear_preparacion(X_train)
    print("Train / validación / prueba:", len(X_train), len(X_validacion), len(X_prueba))

### 4.2 Promedio · línea base
Siempre predice el promedio de entrenamiento. Un modelo útil debería mejorar esta referencia. Ejecuta este bloque si estás resolviendo regresión.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_promedio = Pipeline([("preparacion", clone(preparacion)),
                              ("modelo", DummyRegressor(strategy="mean"))])
    modelo_promedio.fit(X_train, y_train)
    modelos["Promedio"] = modelo_promedio

### 4.3 Regresión lineal
Aprende una combinación lineal de los predictores; es una primera comparación fácil de explicar. Ejecuta este bloque si estás resolviendo regresión.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_lineal = Pipeline([("preparacion", clone(preparacion)),
                              ("modelo", LinearRegression())])
    modelo_lineal.fit(X_train, y_train)
    modelos["Regresión lineal"] = modelo_lineal

### 4.4 Árbol sin límite
Permite observar sobreajuste: compara sus errores de train y validación. Ejecuta este bloque si estás resolviendo regresión.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_arbol_libre = Pipeline([("preparacion", clone(preparacion)),
                              ("modelo", DecisionTreeRegressor(random_state=SEMILLA))])
    modelo_arbol_libre.fit(X_train, y_train)
    modelos["Árbol sin límite"] = modelo_arbol_libre

### 4.5 Árbol podado
Limita profundidad y exige ejemplos por hoja. No se ejecuta una búsqueda de hiperparámetros. Ejecuta este bloque si estás resolviendo regresión.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_arbol_podado = Pipeline([("preparacion", clone(preparacion)),
                              ("modelo", DecisionTreeRegressor(max_depth=5, min_samples_leaf=5, random_state=SEMILLA))])
    modelo_arbol_podado.fit(X_train, y_train)
    modelos["Árbol podado"] = modelo_arbol_podado

### 4.6 Random Forest
Promedia 100 árboles. La profundidad y el tamaño de hoja ya tienen valores moderados. Ejecuta este bloque si estás resolviendo regresión.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_rf = Pipeline([("preparacion", clone(preparacion)),
                              ("modelo", RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1))])
    modelo_rf.fit(X_train, y_train)
    modelos["Random Forest"] = modelo_rf

### 4.7 XGBoost
Suma correcciones sucesivas. Se usan 100 árboles y tasa de aprendizaje 0.05. Ejecuta este bloque si estás resolviendo regresión.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_xgb = Pipeline([("preparacion", clone(preparacion)),
                              ("modelo", XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=SEMILLA, n_jobs=-1))])
    modelo_xgb.fit(X_train, y_train)
    modelos["XGBoost"] = modelo_xgb

### 4.8 LightGBM
Otro boosting; compara su error con RF/XGBoost en las mismas filas de validación. Ejecuta este bloque si estás resolviendo regresión.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_lgb = Pipeline([("preparacion", clone(preparacion)),
                              ("modelo", LGBMRegressor(n_estimators=100, learning_rate=0.05, num_leaves=15, random_state=SEMILLA, n_jobs=-1, verbosity=-1))])
    modelo_lgb.fit(X_train, y_train)
    modelos["LightGBM"] = modelo_lgb

### 4.9 Comparación y sobreajuste
Cada modelo se entrenó una sola vez. Ahora se evalúa en train y validación. Un error muy bajo en train y alto en validación indica revisar sobreajuste. RMSE penaliza más los errores grandes; MAE se interpreta en unidades del target.

**Si el enunciado da una regla actual**, compárala en las mismas filas. Ejemplo de agentes: metricas(y_validacion, X_validacion["retiros_prom_7d"]). En RapiMenu: metricas(y_validacion, 30 + 3 * X_validacion["distancia_km"]). Cambia la fórmula según tu caso; no se ejecuta ninguna regla inventada.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    resultados = []
    for nombre, modelo in modelos.items():
        for conjunto, datos, reales in [("Train", X_train, y_train), ("Validación", X_validacion, y_validacion)]:
            resultados.append({"Modelo": nombre, "Conjunto": conjunto, **metricas(reales, modelo.predict(datos))})
    tabla_comparacion = pd.DataFrame(resultados).set_index(["Modelo", "Conjunto"])
    tabla_validacion = tabla_comparacion.xs("Validación", level="Conjunto")
    display(tabla_comparacion.round(3))
    display(tabla_validacion.sort_values("RMSE").round(3))

## 5. Ruta de series de tiempo
### 5.1 Calendario y memoria histórica
Sigue recaudación y el caso de series: fecha regular → calendario/rezagos → modelos → pronóstico recursivo. El target histórico no entra en la media móvil de su propia fila: se mantiene shift(1), corrigiendo ese problema de las referencias.

calendario(fechas, feriados) usa información conocida de antemano. crear_rezagos(serie, rezagos, ventana, feriados) construye la tabla de entrenamiento. pronosticar(modelo, historia, fechas, rezagos, ventana, feriados) usa sus propias predicciones al avanzar; modelo=None representa persistencia. Ninguna función cambia la historia de entrada.

In [ ]:
def calendario(fechas, feriados=()):
    """Construye variables de calendario conocidas antes de observar el target."""
    indice = pd.DatetimeIndex(fechas)
    dias_feriados = pd.DatetimeIndex(pd.to_datetime(list(feriados), format="%Y-%m-%d", errors="raise"))
    if indice.hasnans or dias_feriados.hasnans:
        raise ValueError("El calendario no admite fechas vacías.")
    return pd.DataFrame({"anio": indice.year, "mes": indice.month, "dia": indice.day,
        "dia_semana": indice.dayofweek, "semana_iso": indice.isocalendar().week.to_numpy(dtype=int),
        "fin_de_semana": (indice.dayofweek >= 5).astype(int),
        "feriado": indice.normalize().isin(dias_feriados.normalize()).astype(int)}, index=indice)


def crear_rezagos(serie, rezagos, ventana, feriados=()):
    """Usa solo valores anteriores; omite las filas iniciales sin memoria completa."""
    datos = calendario(serie.index, feriados)
    for rezago in rezagos:
        datos[f"lag_{rezago}"] = serie.shift(rezago)
    datos["media_movil"] = serie.shift(1).rolling(ventana).mean()
    datos["objetivo"] = serie.to_numpy()
    return datos.dropna()


def pronosticar(modelo, historia, fechas, rezagos, ventana, feriados=()):
    """Pronostica recursivamente sin leer targets futuros ni modificar historia."""
    if len(historia) < max(rezagos + [ventana]):
        raise ValueError("Historia insuficiente para los rezagos y la media móvil.")
    if len(fechas) == 0 or fechas.freq is None:
        raise ValueError("Se requieren fechas futuras con frecuencia explícita.")
    esperadas = pd.date_range(historia.index[-1] + fechas.freq, periods=len(fechas), freq=fechas.freq)
    if not fechas.equals(esperadas):
        raise ValueError("Las fechas futuras deben empezar en el siguiente periodo y ser consecutivas.")
    if not historia.index.equals(pd.date_range(historia.index[0], periods=len(historia), freq=fechas.freq)) or not np.isfinite(historia.to_numpy(dtype=float)).all():
        raise ValueError("La historia debe ser regular, ordenada y finita.")
    valores = historia.to_numpy(dtype=float).tolist()
    filas_calendario = calendario(fechas, feriados)
    predicciones = []
    for paso, fecha in enumerate(fechas):
        fila = filas_calendario.iloc[[paso]].copy()
        for rezago in rezagos:
            fila[f"lag_{rezago}"] = valores[-rezago]
        fila["media_movil"] = np.mean(valores[-ventana:])
        prediccion = valores[-1] if modelo is None else float(modelo.predict(fila)[0])
        if not np.isfinite(prediccion):
            raise ValueError("El modelo produjo un pronóstico no finito.")
        predicciones.append(prediccion)
        valores.append(prediccion)
    return pd.Series(predicciones, index=fechas, name="Predicción")

### 5.2 Ordenar y reservar periodos
Se ordenan las fechas y se exige un valor por periodo, sin duplicados ni huecos. No se inventan valores históricos con relleno automático.

Los últimos HORIZONTE periodos son prueba; los HORIZONTE anteriores, validación. El resto es entrenamiento. Así comparas ventanas de la misma duración que el pronóstico solicitado. Debe quedar historia suficiente para los rezagos.

Los rezagos diarios ya son 1/7/14 y la media usa siete periodos; para otras frecuencias son 1/2/3 y tres periodos. Solo cambia esas líneas si tu enunciado lo requiere. FERIADOS es opcional: incluye también los futuros conocidos.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    if FECHA not in df_limpio or FECHA == TARGET:
        raise ValueError("FECHA debe existir y ser distinta del target.")
    if isinstance(HORIZONTE, bool) or not isinstance(HORIZONTE, int) or HORIZONTE < 1:
        raise ValueError("HORIZONTE debe ser un entero positivo.")
    REZAGOS = [1, 7, 14] if FRECUENCIA == "D" else [1, 2, 3]
    VENTANA = 7 if FRECUENCIA == "D" else 3
    FERIADOS = []  # Opcional: ["2026-01-01", ...], sin calendario nacional automático
    serie = df_limpio.assign(**{FECHA: pd.to_datetime(df_limpio[FECHA], format=FORMATO_FECHA, errors="raise")}).sort_values(FECHA).set_index(FECHA)[TARGET]
    if serie.index.hasnans or not serie.index.is_unique:
        raise ValueError("Hay fechas vacías o duplicadas. Decide cómo corregirlas o agregarlas.")
    if not serie.index.equals(pd.date_range(serie.index[0], serie.index[-1], freq=FRECUENCIA)):
        raise ValueError("Hay huecos o fechas fuera de la frecuencia declarada.")
    if len(serie) - 2 * HORIZONTE <= max(REZAGOS + [VENTANA]) + 5:
        raise ValueError("Historia insuficiente: reduce horizonte/rezagos conscientemente o aporta más datos.")
    historia_train = serie.iloc[:-2 * HORIZONTE].copy()
    y_validacion_temporal = serie.iloc[-2 * HORIZONTE:-HORIZONTE].copy()
    y_prueba_temporal = serie.iloc[-HORIZONTE:].copy()
    fechas_validacion = pd.date_range(y_validacion_temporal.index[0], periods=HORIZONTE, freq=FRECUENCIA)
    tabla_train = crear_rezagos(historia_train, REZAGOS, VENTANA, FERIADOS)
    X_train_temporal, y_train_temporal = tabla_train.drop(columns="objetivo"), tabla_train["objetivo"]
    display(pd.DataFrame([{"Conjunto": nombre, "Periodos": len(s), "Inicio": s.index[0], "Fin": s.index[-1]}
        for nombre, s in [("Train", historia_train), ("Validación", y_validacion_temporal), ("Prueba", y_prueba_temporal)]]))

### 5.3 Persistencia · línea base
Repite el último valor conocido durante todo el horizonte. Sirve para comprobar si los modelos aportan una mejora.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    modelos["Persistencia"] = None

### 5.4 Random Forest
Aprende con calendario y rezagos históricos. En validación recibirá sus propias predicciones como memoria, no valores reales futuros.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    modelo_rf_temporal = RandomForestRegressor(n_estimators=100, max_depth=8, min_samples_leaf=3, random_state=SEMILLA, n_jobs=-1)
    modelo_rf_temporal.fit(X_train_temporal, y_train_temporal)
    modelos["Random Forest"] = modelo_rf_temporal

### 5.5 XGBoost
Es el segundo algoritmo del ejemplo de recaudación. Los dos modelos se comparan con las mismas fechas y el mismo pronóstico recursivo.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    modelo_xgb_temporal = XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=SEMILLA, n_jobs=-1)
    modelo_xgb_temporal.fit(X_train_temporal, y_train_temporal)
    modelos["XGBoost"] = modelo_xgb_temporal

### 5.6 Comparar modelos
Train se mide a un paso con historia real; validación se mide recursivamente. Su brecha mezcla generalización y acumulación de error, por lo que no demuestra solamente sobreajuste. La curva compara la ventana completa.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    resultados = []
    predicciones_validacion = {}
    for nombre, modelo in modelos.items():
        pred_train = historia_train.shift(1).loc[y_train_temporal.index] if modelo is None else modelo.predict(X_train_temporal)
        pred_val = pronosticar(modelo, historia_train, fechas_validacion, REZAGOS, VENTANA, FERIADOS)
        predicciones_validacion[nombre] = pred_val
        resultados.extend([{"Modelo": nombre, "Conjunto": "Train un paso", **metricas(y_train_temporal, pred_train)},
                           {"Modelo": nombre, "Conjunto": "Validación", **metricas(y_validacion_temporal, pred_val)}])
    tabla_comparacion = pd.DataFrame(resultados).set_index(["Modelo", "Conjunto"])
    tabla_validacion = tabla_comparacion.xs("Validación", level="Conjunto")
    display(tabla_comparacion.round(3))
    pd.DataFrame({"Real": y_validacion_temporal, **predicciones_validacion}).plot(figsize=(12, 4), title="Validación recursiva")
    plt.show()

## 6. Elección, prueba e interpretación
### 6.1 Elegir con validación
Deja MODELO_ELEGIDO="auto" para escoger menor RMSE en regresión o menor MAE recursivo en series. Si el enunciado pide otro modelo, copia su nombre de la tabla. La línea base también puede ganar.

Solo después de elegir se evalúa prueba. No cambies de modelo mirando ese resultado; eso convertiría prueba en otra validación.

In [ ]:
MODELO_ELEGIDO = "auto"  # Opcional: "Regresión lineal", "Random Forest", "XGBoost", etc.
if TIPO_PROBLEMA in {"regresion", "serie_tiempo"}:
    criterio = "RMSE" if TIPO_PROBLEMA == "regresion" else "MAE"
    nombre_elegido = tabla_validacion[criterio].idxmin() if MODELO_ELEGIDO == "auto" else MODELO_ELEGIDO
    if nombre_elegido not in modelos:
        raise ValueError("Copia un nombre de la tabla de modelos ejecutados.")
    modelo_elegido = modelos[nombre_elegido]
    print("Modelo:", nombre_elegido, "| criterio:", criterio, "de validación")

### 6.2 Evaluación final
Se reentrena únicamente el modelo elegido con train+validación y se evalúa la reserva de prueba una vez. En series se pronostica toda la ventana sin incorporar sus targets reales.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    modelo_final = clone(modelo_elegido)
    modelo_final.fit(X_desarrollo, y_desarrollo)
    pred_prueba = modelo_final.predict(X_prueba)
    metricas_prueba = metricas(y_prueba, pred_prueba)
    display(pd.Series(metricas_prueba, name="Prueba final").to_frame())

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    historia_desarrollo = serie.iloc[:-HORIZONTE].copy()
    tabla_desarrollo = crear_rezagos(historia_desarrollo, REZAGOS, VENTANA, FERIADOS)
    modelo_final = clone(modelo_elegido).fit(tabla_desarrollo.drop(columns="objetivo"), tabla_desarrollo["objetivo"]) if modelo_elegido is not None else None
    fechas_prueba = pd.date_range(y_prueba_temporal.index[0], periods=HORIZONTE, freq=FRECUENCIA)
    pred_prueba_temporal = pronosticar(modelo_final, historia_desarrollo, fechas_prueba, REZAGOS, VENTANA, FERIADOS)
    metricas_prueba = metricas(y_prueba_temporal, pred_prueba_temporal)
    display(pd.Series(metricas_prueba, name="Prueba recursiva final").to_frame())

### 6.3 ¿Dónde se equivoca?
En regresión, revisa diagonal real/predicho, residuos y mayores errores; un residuo positivo significa subestimación. En series, revisa la curva de prueba, errores por día de semana y distancia desde el origen. Estos gráficos describen el resultado final; no se usan para volver a elegir.

Como en RapiMenu/agentes, puedes agrupar error_absoluto por una columna relevante para responder una pregunta de segmentos.

In [ ]:
if TIPO_PROBLEMA == "regresion":
    diagnostico = df_limpio.loc[y_prueba.index].assign(prediccion=pred_prueba,
        residuo=y_prueba.to_numpy() - pred_prueba, error_absoluto=np.abs(y_prueba.to_numpy() - pred_prueba))
    fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
    ejes[0].scatter(y_prueba, pred_prueba, alpha=0.5)
    limites = [min(y_prueba.min(), pred_prueba.min()), max(y_prueba.max(), pred_prueba.max())]
    ejes[0].plot(limites, limites, "--", color="gray")
    ejes[0].set(xlabel="Real", ylabel="Predicción")
    ejes[1].scatter(pred_prueba, diagnostico["residuo"], alpha=0.5)
    ejes[1].axhline(0, color="gray", linestyle="--")
    ejes[1].set(xlabel="Predicción", ylabel="Real menos predicción")
    sns.histplot(diagnostico["residuo"], ax=ejes[2])
    plt.tight_layout()
    plt.show()
    display(diagnostico.sort_values("error_absoluto", ascending=False).head(5))
    # Ejemplo si existe zona: display(diagnostico.groupby("zona")["error_absoluto"].agg(["count", "mean"]))

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    diagnostico = pd.DataFrame({"Real": y_prueba_temporal.to_numpy(), "Predicción": pred_prueba_temporal.to_numpy()}, index=fechas_prueba)
    diagnostico[["Real", "Predicción"]].plot(figsize=(12, 4), title="Prueba recursiva final")
    plt.show()
    diagnostico["paso"] = np.arange(1, len(diagnostico) + 1)
    diagnostico["error_absoluto"] = np.abs(diagnostico["Real"] - diagnostico["Predicción"])
    display(diagnostico)
    display(diagnostico.groupby(diagnostico.index.dayofweek)["error_absoluto"].agg(["count", "mean"]))

### 6.4 Principales variables
Los árboles muestran importancia interna; la regresión lineal muestra magnitud de coeficientes sobre las variables preparadas. Son medidas diferentes y no demuestran causalidad. La línea base no tiene variables explicativas. Usa las tres primeras para redactar una interpretación breve.

In [ ]:
if TIPO_PROBLEMA in {"regresion", "serie_tiempo"}:
    estimador = modelo_final.named_steps["modelo"] if TIPO_PROBLEMA == "regresion" else modelo_final
    nombres_variables = modelo_final.named_steps["preparacion"].get_feature_names_out() if TIPO_PROBLEMA == "regresion" else X_train_temporal.columns
    if hasattr(estimador, "feature_importances_") or hasattr(estimador, "coef_"):
        valores_importancia = estimador.feature_importances_ if hasattr(estimador, "feature_importances_") else np.abs(estimador.coef_)
        importancias = pd.Series(valores_importancia, index=nombres_variables).sort_values(ascending=False)
        display(importancias.head(12))
        importancias.head(12).sort_values().plot.barh(title=f"Variables de {nombre_elegido}", figsize=(8, 4))
        plt.show()
    else:
        print("La línea base no tiene importancias ni coeficientes.")

## 7. Predicción y entrega
### 7.1 ¿Qué te pide el enunciado?
**Regresión:** si entregan otra base, cambia RUTA_BASE_NUEVA. Con None se omite; no se necesita target en esa base.

**Series:** se generan los siguientes HORIZONTE periodos. Si piden hasta una fecha final, usa FECHA_FIN_PRONOSTICO. Para entregar solo un intervalo, indica INICIO_REPORTE/FIN_REPORTE; se calculan los pasos intermedios antes de filtrar. Las fechas deben estar dentro de tu pronóstico.

Los resultados van a /content/salidas_guia_pc2. No necesitas guardar y recargar el modelo para predecir dentro de esta misma sesión.

In [ ]:
RUTA_BASE_NUEVA = None  # Ejemplo: "/content/agentes_nuevos.csv"
FECHA_FIN_PRONOSTICO = None  # Solo series; ejemplo: "2026-01-31"
INICIO_REPORTE = None  # Solo series; None usa el primer periodo futuro
FIN_REPORTE = None  # Solo series; None usa el último periodo futuro
SALIDAS = Path("/content/salidas_guia_pc2")
SALIDAS.mkdir(parents=True, exist_ok=True)

### 7.2 Otra base de regresión
Después de reportar prueba, se ajusta una copia del modelo elegido con toda la base disponible para uso operativo, como en agentes/RapiMenu. preparar_datos y el Pipeline se reutilizan; la predicción en sí no vuelve a aprender nada. El archivo conserva sus filas y añade ml_prediccion.

In [ ]:
if TIPO_PROBLEMA == "regresion" and RUTA_BASE_NUEVA is not None:
    df_nueva_original = leer_archivo(RUTA_BASE_NUEVA)
    df_nueva = preparar_datos(df_nueva_original)
    if set(X.columns) - set(df_nueva.columns):
        raise ValueError("Faltan predictores en la base nueva. Conserva nombres y unidades.")
    X_nueva = df_nueva.loc[:, X.columns]
    modelo_operativo = clone(modelo_elegido).fit(X, y)
if TIPO_PROBLEMA == "regresion" and RUTA_BASE_NUEVA is None:
    print("Sin segunda base: ya puedes entregar la evaluación y conclusiones.")

In [ ]:
if TIPO_PROBLEMA == "regresion" and RUTA_BASE_NUEVA is not None:
    resultado_nuevo = df_nueva_original.assign(ml_prediccion=modelo_operativo.predict(X_nueva))
    resultado_nuevo.to_csv(SALIDAS / "predicciones_base_nueva.csv", index=False, encoding="utf-8-sig")
    display(resultado_nuevo.head(10))
    print("Exportado:", SALIDAS / "predicciones_base_nueva.csv")

### 7.3 Pronóstico operativo
Después de medir prueba se reentrena con todo el histórico. Se pronostica desde el periodo siguiente a la última fecha, sin respuestas futuras. La fecha final debe coincidir con la frecuencia. No se recortan valores negativos automáticamente: eso cambiaría el modelo que evaluaste.

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    historia_operativa = serie.copy()
    tabla_operativa = crear_rezagos(historia_operativa, REZAGOS, VENTANA, FERIADOS)
    modelo_operativo = clone(modelo_elegido).fit(tabla_operativa.drop(columns="objetivo"), tabla_operativa["objetivo"]) if modelo_elegido is not None else None

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    inicio_futuro = historia_operativa.index[-1] + pd.tseries.frequencies.to_offset(FRECUENCIA)
    fin_futuro = pd.Timestamp(FECHA_FIN_PRONOSTICO) if FECHA_FIN_PRONOSTICO is not None else None
    if fin_futuro is not None and (pd.isna(fin_futuro) or fin_futuro < inicio_futuro):
        raise ValueError("La fecha final debe ser posterior al origen del pronóstico.")
    fechas_futuras = pd.date_range(start=inicio_futuro, end=fin_futuro, periods=HORIZONTE if fin_futuro is None else None, freq=FRECUENCIA)
    if fin_futuro is not None and (len(fechas_futuras) == 0 or fechas_futuras[-1] != fin_futuro):
        raise ValueError("La fecha final no coincide con un periodo de la frecuencia.")
    pred_futuro = pronosticar(modelo_operativo, historia_operativa, fechas_futuras, REZAGOS, VENTANA, FERIADOS)
    pronostico = pd.DataFrame({FECHA: fechas_futuras, "ml_prediccion": pred_futuro.to_numpy()})
    inicio_reporte = pd.Timestamp(INICIO_REPORTE) if INICIO_REPORTE is not None else fechas_futuras[0]
    fin_reporte = pd.Timestamp(FIN_REPORTE) if FIN_REPORTE is not None else fechas_futuras[-1]
    if pd.isna(inicio_reporte) or pd.isna(fin_reporte) or not (fechas_futuras[0] <= inicio_reporte <= fin_reporte <= fechas_futuras[-1]):
        raise ValueError("El intervalo debe estar ordenado y dentro del horizonte pronosticado.")
    reporte = pronostico.loc[pronostico[FECHA].between(inicio_reporte, fin_reporte)].copy()
    if reporte.empty:
        raise ValueError("El intervalo no contiene periodos de la frecuencia.")
    pronostico.to_csv(SALIDAS / "pronostico_completo.csv", index=False, encoding="utf-8-sig")
    reporte.to_csv(SALIDAS / "pronostico_intervalo.csv", index=False, encoding="utf-8-sig")
    pronostico.plot(x=FECHA, y="ml_prediccion", figsize=(12, 4), title="Pronóstico futuro")
    plt.show()
    display(reporte)

### 7.4 Resumen mensual
Como en recaudación, se agregan los valores del intervalo. Usa sum para transacciones/ventas acumulables y mean para niveles como precios. resumir_mensual(serie, frecuencia, agregacion) informa cobertura y mes_completo; no extrapola meses parciales ni convierte huecos en ceros.

La comparación interanual requiere ambos meses completos y un histórico distinto de cero. Sumar predicciones no elimina la incertidumbre recursiva.

In [ ]:
def resumir_mensual(serie, frecuencia, agregacion):
    """Agrega sin rellenar huecos; marca cobertura según la frecuencia de la serie."""
    if agregacion not in {"sum", "mean"}:
        raise ValueError("La agregación mensual debe ser sum o mean.")
    if serie.empty or not isinstance(serie.index, pd.DatetimeIndex) or not serie.index.is_unique or serie.index.hasnans:
        raise ValueError("El resumen mensual requiere fechas únicas y no vacías.")
    ordenada = serie.sort_index()
    inicio_mes = ordenada.index[0].to_period("M").start_time
    fin_mes = ordenada.index[-1].to_period("M").end_time
    # Extiende la misma cuadrícula en ambas direcciones para medir meses parciales.
    desplazamiento = pd.tseries.frequencies.to_offset(frecuencia)
    if isinstance(desplazamiento, pd.tseries.offsets.Tick):
        inicio_alineado = ordenada.index[0] - ((ordenada.index[0] - inicio_mes) // pd.Timedelta(desplazamiento)) * desplazamiento
        anteriores = pd.date_range(start=inicio_alineado, end=ordenada.index[0], freq=frecuencia)
    else:
        anteriores_amplias = pd.date_range(end=ordenada.index[0], periods=32, freq=frecuencia)
        anteriores = anteriores_amplias[anteriores_amplias >= inicio_mes]
    posteriores = pd.date_range(start=ordenada.index[0], end=fin_mes, freq=frecuencia)
    esperadas = anteriores.union(posteriores)
    if len(ordenada.index.difference(esperadas)):
        raise ValueError("Las fechas del resumen no coinciden con la frecuencia declarada.")
    meses = pd.period_range(inicio_mes, fin_mes, freq="M")
    grupos = ordenada.groupby(ordenada.index.to_period("M"))
    valores = grupos.sum(min_count=1) if agregacion == "sum" else grupos.mean()
    conteo_esperado = pd.Series(1, index=esperadas).groupby(esperadas.to_period("M")).sum()
    resumen = pd.DataFrame({"valor": valores, "periodos_observados": grupos.count(),
        "periodos_esperados": conteo_esperado}).reindex(meses)
    conteos = resumen.assign(periodos_observados=resumen["periodos_observados"].fillna(0).astype(int),
        periodos_esperados=resumen["periodos_esperados"].fillna(0).astype(int))
    return conteos.assign(cobertura_pct=conteos["periodos_observados"].div(conteos["periodos_esperados"].replace(0, np.nan)).mul(100),
        mes_completo=conteos["periodos_observados"].eq(conteos["periodos_esperados"]) & conteos["periodos_esperados"].gt(0)).rename_axis("mes")

In [ ]:
if TIPO_PROBLEMA == "serie_tiempo":
    AGREGACION_MENSUAL = "sum"  # Cambia a "mean" si tu objetivo es un nivel, no una cantidad acumulable
    resumen_mensual = resumir_mensual(reporte.set_index(FECHA)["ml_prediccion"], FRECUENCIA, AGREGACION_MENSUAL)
    mensual_historico = resumir_mensual(serie, FRECUENCIA, AGREGACION_MENSUAL)
    resumen_mensual["real_anio_anterior"] = [mensual_historico.loc[mes - 12, "valor"]
        if completo and mes - 12 in mensual_historico.index and mensual_historico.loc[mes - 12, "mes_completo"] else np.nan
        for mes, completo in resumen_mensual["mes_completo"].items()]
    resumen_mensual["variacion_interanual_pct"] = (resumen_mensual["valor"] / resumen_mensual["real_anio_anterior"].replace(0, np.nan) - 1) * 100
    resumen_mensual.to_csv(SALIDAS / "pronostico_mensual.csv", encoding="utf-8-sig")
    display(resumen_mensual)

## Cierre: respuesta para tu PC2
Escribe: **qué predices y en qué unidades; qué corregiste; dos hallazgos; modelo y criterio de validación; métricas de prueba; variables/errores relevantes; predicción solicitada y limitaciones**. Si hay una regla de negocio, compara sus errores con los del modelo sobre las mismas filas.

Guarda el notebook y descarga los CSV del panel Archivos antes de terminar la sesión. No hace falta ejecutar los anexos para completar regresión o series.

**Referencias principales:** Caso_RapiMenu_regresion.ipynb, PD_Solucion_Regresion_Series.ipynb y Caso2_Series_de_Tiempo_Entidad_X.ipynb, en referencias/. Se conserva su estructura didáctica; la selección usa validación y las medias móviles de entrenamiento se desplazan para evitar fuga del target.

# Anexos opcionales · Fuera del recorrido de 110 minutos
No necesitas estos bloques para entregar regresión o series. Las banderas empiezan en False y las instalaciones están dentro de sus condiciones. Para una búsqueda que quieras adoptar, ejecuta su anexo antes de fijar la elección/evaluar prueba; no vuelvas a elegir mirando prueba. Sus resultados no reemplazan automáticamente el modelo principal.

## Anexo A · Clasificación de apoyo
Solo se ejecuta si TIPO_PROBLEMA="clasificacion". Sigue S4/S5: logística, árbol y Random Forest; train/validación/prueba estratificados, F1 macro para comparar y matriz de confusión. Los códigos de etiquetas se aprenden con train y se traducen al exportar. No requiere una instalación adicional.

In [ ]:
#@title Anexo A · Solo para TIPO_PROBLEMA = "clasificacion"
if TIPO_PROBLEMA == "clasificacion":
    from sklearn.preprocessing import LabelEncoder
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
    X_clf = df_limpio.drop(columns=[TARGET] + COLUMNAS_EXCLUIR).select_dtypes(exclude=["datetime", "datetimetz", "timedelta"])
    y_clf = df_limpio[TARGET]
    if X_clf.empty or y_clf.nunique() < 2 or y_clf.value_counts().min() < 5:
        raise ValueError("Clasificación necesita predictores y al menos dos clases con cinco filas por clase.")
    X_des_clf, X_prueba_clf, y_des_clf, y_prueba_clf = train_test_split(X_clf, y_clf, test_size=0.20, random_state=SEMILLA, stratify=y_clf)
    X_train_clf, X_val_clf, y_tr_etiquetas, y_val_etiquetas = train_test_split(X_des_clf, y_des_clf, test_size=0.25, random_state=SEMILLA, stratify=y_des_clf)
    codificador = LabelEncoder().fit(y_tr_etiquetas)
    y_train_clf, y_val_clf = codificador.transform(y_tr_etiquetas), codificador.transform(y_val_etiquetas)
    modelos_clf = {}
    resultados_clf = []
    for nombre, estimador_clf in {"Logística": LogisticRegression(max_iter=1000),
            "Árbol": DecisionTreeClassifier(max_depth=5, random_state=SEMILLA),
            "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEMILLA, n_jobs=-1)}.items():
        pipeline_clf = Pipeline([("preparacion", crear_preparacion(X_train_clf)), ("modelo", estimador_clf)])
        pipeline_clf.fit(X_train_clf, y_train_clf)
        modelos_clf[nombre] = pipeline_clf
        for conjunto, datos, reales in [("Train", X_train_clf, y_train_clf), ("Validación", X_val_clf, y_val_clf)]:
            pred_clf = pipeline_clf.predict(datos)
            resultados_clf.append({"Modelo": nombre, "Conjunto": conjunto, "Accuracy": accuracy_score(reales, pred_clf),
                                   "F1_macro": f1_score(reales, pred_clf, average="macro", zero_division=0)})
    tabla_clf = pd.DataFrame(resultados_clf).set_index(["Modelo", "Conjunto"])
    display(tabla_clf.round(3))
    nombre_elegido = tabla_clf.xs("Validación", level="Conjunto")["F1_macro"].idxmax() if MODELO_ELEGIDO == "auto" else MODELO_ELEGIDO
    if nombre_elegido not in modelos_clf:
        raise ValueError("Copia un nombre de la tabla de clasificación.")
    modelo_final = clone(modelos_clf[nombre_elegido]).fit(X_des_clf, codificador.transform(y_des_clf))
    pred_prueba_clf = modelo_final.predict(X_prueba_clf)
    print("Modelo:", nombre_elegido)
    print(classification_report(codificador.transform(y_prueba_clf), pred_prueba_clf,
                               labels=np.arange(len(codificador.classes_)), target_names=[str(c) for c in codificador.classes_], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(codificador.transform(y_prueba_clf), pred_prueba_clf,
        labels=np.arange(len(codificador.classes_)), display_labels=codificador.classes_)
    plt.show()
    if RUTA_BASE_NUEVA is not None:
        modelo_operativo = clone(modelos_clf[nombre_elegido]).fit(X_clf, codificador.transform(y_clf))
        nuevos_clf = leer_archivo(RUTA_BASE_NUEVA)
        pred_nuevos_clf = modelo_operativo.predict(preparar_datos(nuevos_clf).loc[:, X_clf.columns])
        nuevos_clf.assign(ml_prediccion=codificador.inverse_transform(pred_nuevos_clf)).to_csv(SALIDAS / "predicciones_base_nueva.csv", index=False, encoding="utf-8-sig")

## Anexo B · Búsqueda pequeña con validación cruzada
Aplica a RF tabular, no a series. GridSearchCV recibe el Pipeline completo: cada fold aprende su propia preparación. La búsqueda usa solo train; no modifica la elección ni consume prueba. Compara después el candidato en validación si vas a adoptarlo.

In [ ]:
#@title Anexo B · Búsqueda opcional
EJECUTAR_BUSQUEDA = False
if EJECUTAR_BUSQUEDA and TIPO_PROBLEMA in {"regresion", "clasificacion"}:
    from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold, StratifiedKFold
    es_regresion = TIPO_PROBLEMA == "regresion"
    candidato = modelos["Random Forest"] if es_regresion else modelos_clf["Random Forest"]
    cv = KFold(3, shuffle=True, random_state=SEMILLA) if es_regresion else StratifiedKFold(3, shuffle=True, random_state=SEMILLA)
    TIPO_BUSQUEDA = "grid"  # "random" para probar una muestra de configuraciones
    grilla = {"modelo__max_depth": [4, 8], "modelo__min_samples_leaf": [3, 6]}
    opciones = dict(scoring="neg_root_mean_squared_error" if es_regresion else "f1_macro", cv=cv, n_jobs=-1, error_score="raise")
    busqueda = GridSearchCV(clone(candidato), grilla, **opciones) if TIPO_BUSQUEDA == "grid" else RandomizedSearchCV(clone(candidato), grilla, n_iter=3, random_state=SEMILLA, **opciones)
    busqueda.fit(X_train if es_regresion else X_train_clf, y_train if es_regresion else y_train_clf)
    print(busqueda.best_params_)
    display(pd.DataFrame(busqueda.cv_results_)[["params", "mean_test_score", "rank_test_score"]])

## Anexo C · Backtesting de series
Solo si te piden varios orígenes. Repite entrenamientos dentro de historia_train y mantiene prueba reservada. Las ventanas no se solapan por defecto. Requiere más historia que la validación simple.

In [ ]:
#@title Anexo C · Backtesting opcional
EJECUTAR_BACKTEST = False
if EJECUTAR_BACKTEST and TIPO_PROBLEMA == "serie_tiempo":
    VENTANAS_BACKTEST = 3
    origenes = [len(historia_train) - HORIZONTE * k for k in range(VENTANAS_BACKTEST, 0, -1)]
    if min(origenes) <= max(REZAGOS + [VENTANA]) + 5:
        raise ValueError("No alcanza historia para tres ventanas. Reduce ventanas/horizonte o aporta más datos.")
    filas_backtest = []
    for numero, origen in enumerate(origenes, start=1):
        historia_ventana = historia_train.iloc[:origen]
        datos_ventana = crear_rezagos(historia_ventana, REZAGOS, VENTANA, FERIADOS)
        fechas_ventana = pd.date_range(historia_train.index[origen], periods=HORIZONTE, freq=FRECUENCIA)
        for nombre, modelo in modelos.items():
            ajustado = clone(modelo).fit(datos_ventana.drop(columns="objetivo"), datos_ventana["objetivo"]) if modelo is not None else None
            pred = pronosticar(ajustado, historia_ventana, fechas_ventana, REZAGOS, VENTANA, FERIADOS)
            filas_backtest.append({"Ventana": numero, "Modelo": nombre, "Origen": historia_ventana.index[-1],
                                   **metricas(historia_train.loc[fechas_ventana], pred)})
    tabla_backtest = pd.DataFrame(filas_backtest)
    display(tabla_backtest)
    display(tabla_backtest.groupby("Modelo")[["MAE", "RMSE", "MAPE_pct"]].mean())

## Anexo D · AutoML tabular
LazyPredict compara muchos modelos; puede tardar. FLAML busca con un presupuesto de 30 segundos. Ambos usan exclusivamente train y validación, con preparación ajustada en train. No se aplican a la ruta temporal.

Para simplificar FLAML sin un adaptador de clases, se proporciona validación explícita y se usa holdout, no CV sobre una matriz preparada globalmente. [Documentación de FLAML](https://microsoft.github.io/FLAML/docs/Use-Cases/Task-Oriented-AutoML/#resampling-strategy). Los resultados no sustituyen automáticamente el modelo elegido.

In [ ]:
#@title Anexo D1 · LazyPredict opcional
EJECUTAR_LAZY = False
if EJECUTAR_LAZY and TIPO_PROBLEMA in {"regresion", "clasificacion"}:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lazypredict==0.4.0"])
    from lazypredict.Supervised import LazyRegressor, LazyClassifier
    es_regresion = TIPO_PROBLEMA == "regresion"
    datos_tr, datos_val = (X_train, X_validacion) if es_regresion else (X_train_clf, X_val_clf)
    respuesta_tr, respuesta_val = (y_train, y_validacion) if es_regresion else (y_train_clf, y_val_clf)
    preparacion_lazy = crear_preparacion(datos_tr).fit(datos_tr, respuesta_tr)
    comparador_lazy = LazyRegressor(verbose=0) if es_regresion else LazyClassifier(verbose=0)
    tabla_lazy, _ = comparador_lazy.fit(preparacion_lazy.transform(datos_tr), preparacion_lazy.transform(datos_val), respuesta_tr, respuesta_val)
    display(tabla_lazy)

In [ ]:
#@title Anexo D2 · FLAML opcional
EJECUTAR_FLAML = False
if EJECUTAR_FLAML and TIPO_PROBLEMA in {"regresion", "clasificacion"}:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "flaml[automl]==2.7.0"])
    from flaml import AutoML
    es_regresion = TIPO_PROBLEMA == "regresion"
    datos_tr, datos_val = (X_train, X_validacion) if es_regresion else (X_train_clf, X_val_clf)
    respuesta_tr, respuesta_val = (y_train, y_validacion) if es_regresion else (y_train_clf, y_val_clf)
    preparacion_flaml = crear_preparacion(datos_tr).fit(datos_tr, respuesta_tr)
    automl = AutoML()
    automl.fit(X_train=preparacion_flaml.transform(datos_tr), y_train=respuesta_tr,
        X_val=preparacion_flaml.transform(datos_val), y_val=respuesta_val,
        task="regression" if es_regresion else "classification", metric="rmse" if es_regresion else "log_loss",
        eval_method="holdout", retrain_full=False, time_budget=30, estimator_list=["rf", "extra_tree"], seed=SEMILLA, verbose=0)
    pred_flaml = automl.predict(preparacion_flaml.transform(datos_val))
    print("Familia y parámetros:", automl.best_estimator, automl.best_config)
    display(pd.Series(metricas(respuesta_val, pred_flaml) if es_regresion else {"F1_macro": f1_score(respuesta_val, pred_flaml, average="macro", zero_division=0)}))

## Anexo E · Red neuronal tabular
Arquitectura corta de 32 y 16 neuronas, hasta 30 épocas y parada temprana. La preparación aprende de train y se reutiliza en validación; prueba no se entrega a la red. El anexo informa validación y no reemplaza el modelo principal.

In [ ]:
#@title Anexo E · Red opcional
EJECUTAR_RED = False
if EJECUTAR_RED and TIPO_PROBLEMA in {"regresion", "clasificacion"}:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow>=2.18,<2.21"])
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEMILLA)
    es_regresion = TIPO_PROBLEMA == "regresion"
    datos_tr, datos_val = (X_train, X_validacion) if es_regresion else (X_train_clf, X_val_clf)
    respuesta_tr, respuesta_val = (y_train, y_validacion) if es_regresion else (y_train_clf, y_val_clf)
    preparacion_red = crear_preparacion(datos_tr).fit(datos_tr, respuesta_tr)
    matriz_tr = preparacion_red.transform(datos_tr).astype(np.float32)
    matriz_val = preparacion_red.transform(datos_val).astype(np.float32)
    unidades = 1 if es_regresion else len(codificador.classes_)
    red = tf.keras.Sequential([tf.keras.layers.Input(shape=(matriz_tr.shape[1],)),
        tf.keras.layers.Dense(32, activation="relu"), tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(unidades, activation="linear" if es_regresion else "softmax")])
    red.compile(optimizer="adam", loss="mse" if es_regresion else "sparse_categorical_crossentropy")
    historial_red = red.fit(matriz_tr, np.asarray(respuesta_tr), validation_data=(matriz_val, np.asarray(respuesta_val)),
        epochs=30, batch_size=32, callbacks=[tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)], verbose=0)
    salida_red = red.predict(matriz_val, verbose=0)
    pred_red = salida_red.ravel() if es_regresion else salida_red.argmax(axis=1)
    print("Épocas:", len(historial_red.history["loss"]))
    display(pd.Series(metricas(respuesta_val, pred_red) if es_regresion else {"F1_macro": f1_score(respuesta_val, pred_red, average="macro", zero_division=0)}))

## Anexo F · SHAP para regresión y series
Explica una salida del modelo final; no demuestra causalidad. Se limita a cinco ejemplos y 20 filas de referencia. En series explica un paso con sus predictores, no toda la cadena recursiva. La línea base se omite.

In [ ]:
#@title Anexo F · SHAP opcional
EJECUTAR_SHAP = False
if EJECUTAR_SHAP and TIPO_PROBLEMA in {"regresion", "serie_tiempo"} and modelo_final is not None:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap>=0.46,<0.50"])
    import shap
    if TIPO_PROBLEMA == "regresion":
        preparacion_shap = modelo_final.named_steps["preparacion"]
        fondo_shap = preparacion_shap.transform(X_train.head(20))
        muestra_shap = preparacion_shap.transform(X_prueba.head(5))
        estimador_shap = modelo_final.named_steps["modelo"]
        nombres_shap = preparacion_shap.get_feature_names_out()
    else:
        fondo_shap, muestra_shap = X_train_temporal.head(20), X_train_temporal.tail(5)
        estimador_shap, nombres_shap = modelo_final, X_train_temporal.columns
    explicador = shap.Explainer(estimador_shap.predict, fondo_shap, algorithm="permutation", feature_names=nombres_shap, seed=SEMILLA)
    valores_shap = explicador(muestra_shap, max_evals=2 * len(nombres_shap) + 1)
    shap.plots.bar(valores_shap)
    shap.plots.waterfall(valores_shap[0])

## Anexo G · Guardar y recargar el modelo
Opcional para reutilizarlo en otra sesión. Se guarda el Pipeline y la configuración mínima; para series, también el origen y la historia que corresponde a ese modelo. Se usa el modelo operativo si ya lo generaste. Solo carga archivos joblib de origen conocido.

Conserva también este notebook: si añadiste reglas o variables a preparar_datos, debes aplicarlas a datos crudos antes de usar el Pipeline recargado.

In [ ]:
#@title Anexo G · Guardado opcional
GUARDAR_MODELO = False
if GUARDAR_MODELO:
    import joblib
    usa_operativo = TIPO_PROBLEMA == "serie_tiempo" or RUTA_BASE_NUEVA is not None
    modelo_guardado = modelo_operativo if usa_operativo else modelo_final
    paquete = {"tipo": TIPO_PROBLEMA, "target": TARGET, "modelo": modelo_guardado, "nombre": nombre_elegido,
               "exclusiones": COLUMNAS_EXCLUIR}
    if TIPO_PROBLEMA == "serie_tiempo":
        paquete.update(historia=historia_operativa, frecuencia=FRECUENCIA, fecha=FECHA,
                       rezagos=REZAGOS, ventana=VENTANA, feriados=FERIADOS)
    else:
        paquete["columnas"] = X.columns.tolist() if TIPO_PROBLEMA == "regresion" else X_clf.columns.tolist()
        if TIPO_PROBLEMA == "clasificacion":
            paquete["codificador"] = codificador
    ruta_modelo = SALIDAS / "modelo_pc2.joblib"
    joblib.dump(paquete, ruta_modelo)
    paquete_recargado = joblib.load(ruta_modelo)
    print("Guardado y recargado:", ruta_modelo)

## Comprobaciones de esta entrega

<details><summary>Verificación técnica · No necesitas ejecutarla durante el examen</summary>

El recorrido principal tiene 406 líneas de código, frente a las 1.631 de la guía anterior; los anexos están separados al final. Se entrega sin datos ni salidas precalculadas.

Se ejecutó tests/verificar_series_pc2.py con Python 3.12.14, NumPy 2.3.5 y pandas 3.0.1 del entorno disponible. Pasaron: sintaxis/estructura, anexos apagados sin importaciones ni instalaciones, CSV con coma/punto y coma y Excel, conservación de entradas, reservas cronológicas, historia insuficiente, huecos/duplicados, calendario y feriados, medias móviles sin el target actual, recursión, persistencia, selección automática/manual, fechas D/W/MS, intervalo posterior al pronóstico, exportaciones y cobertura/comparación mensual. Los gráficos se omitieron en esta prueba ligera.

La prueba completa tests/verificar_guia_pc2.py está actualizada con los siete modelos de regresión, los tres temporales, clasificación binaria/multiclase, gráficos, categorías nuevas, búsquedas, backtesting y guardado/recarga. **No pudo ejecutarse:** faltan nbformat y las librerías de ML en el entorno accesible, y se rechazó la solicitud de descargar dependencias. No se ejecutó Google Colab ni se midió el tiempo real de entrenamiento de tu base. Las versiones locales de NumPy/pandas usadas en la prueba ligera difieren de los rangos de instalación del notebook.

La instalación exige XGBoost >=2.1.4 y LightGBM >=4.6 para su compatibilidad documentada con scikit-learn 1.6: [XGBoost](https://xgboost.readthedocs.io/en/release_3.0.0/changes/v2.1.0.html) y [LightGBM](https://github.com/lightgbm-org/LightGBM/releases/tag/v4.6.0). Esta revisión documental no sustituye la ejecución completa.

</details>